### 1. Column transformer

* [make_column_transformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.make_column_transformer.html#sklearn.compose.make_column_transformer)

In [2]:
import pandas as pd

df = pd.read_csv("http://bit.ly/kaggletrain", nrows=6)
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [3]:
cols = ["Fare", "Embarked", "Sex", "Age"]
X = df[cols]
X

,Fare,Embarked,Sex,Age
0,7.2500,S,male,22.0
1,71.2833,C,female,38.0
2,7.9250,S,female,26.0
3,53.1000,S,female,35.0
4,8.0500,S,male,35.0
5,8.4583,Q,male,NaN


In [5]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import make_column_transformer

In [8]:
ohe = OneHotEncoder()
imp = SimpleImputer()

In [9]:
ct = make_column_transformer(
    (ohe, ["Embarked", "Sex"]),   # apply OneHotEncoder to Embarked and Sex
    (imp, ["Age"]),               # apply SimpleImputer to Age
    remainder="passthrough",)     # include remaining column (Fare) in the output

In [10]:
ct.fit_transform(X)

array([[ 0.    ,  0.    ,  1.    ,  0.    ,  1.    , 22.    ,  7.25  ],
       [ 1.    ,  0.    ,  0.    ,  1.    ,  0.    , 38.    , 71.2833],
       [ 0.    ,  0.    ,  1.    ,  1.    ,  0.    , 26.    ,  7.925 ],
       [ 0.    ,  0.    ,  1.    ,  1.    ,  0.    , 35.    , 53.1   ],
       [ 0.    ,  0.    ,  1.    ,  0.    ,  1.    , 35.    ,  8.05  ],
       [ 0.    ,  1.    ,  0.    ,  0.    ,  1.    , 31.2   ,  8.4583]])

### 2. Select columns to transform

* [make_column_selector](https://scikit-learn.org/stable/modules/generated/sklearn.compose.make_column_selector.html#sklearn.compose.make_column_selector)

1. column name
2. integer position
3. slice
4. boolean mask
5. regex pattern
6. dtypes to include
7. dtypes to exclude

In [11]:
from sklearn.compose import make_column_selector

In [14]:
X.columns

Index(['Fare', 'Embarked', 'Sex', 'Age'], dtype='object')

In [15]:
ct = make_column_transformer((ohe, ["Embarked", "Sex"]))
ct = make_column_transformer((ohe, [1, 2]))               # by integer position
ct = make_column_transformer((ohe, slice(1, 3)))          # by slice
ct = make_column_transformer((ohe, [True, False, True, False]))   # by boolean mask
ct = make_column_transformer((ohe, r"^(S|E).*"))                  # by regex pattern
ct = make_column_transformer((ohe, make_column_selector(dtype_include=object))) # by dtype to include object
ct = make_column_transformer((ohe, make_column_selector(dtype_exclude=float)))  # by dtype to exclude float  

In [16]:
# one-hot encode Embarked and Sex (and drop all other columns)
ct.fit_transform(X)

array([[0., 0., 1., 0., 1.],
       [1., 0., 0., 1., 0.],
       [0., 0., 1., 1., 0.],
       [0., 0., 1., 1., 0.],
       [0., 0., 1., 0., 1.],
       [0., 1., 0., 0., 1.]])

### 4. fit_transform

Use `fit_transform()` on training data, but `transform()` (only) on testing/new data.

Applies the same transformations to both sets of data, which creates consistent columns and prevents data leakage!

### 6. Encode categorical features

1. Binary encoder
2. [OneHotEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html#sklearn.preprocessing.OneHotEncoder) for unordered (nominal) data
3. [pd.get_dummies()](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)
4. [OrdinalEncoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html#sklearn.preprocessing.OrdinalEncoder) for ordered (ordinal) data

In [41]:
import pandas as pd

X = pd.DataFrame(
    {
        "Smoker": ["yes", "no", "no", "yes"],
        "Shape": ["square", "square", "oval", "circle"],
        "Region": ["north", "south", "east", "west"],
        "Class": ["third", "first", "second", "third"],
        "Size": ["S", "S", "L", "XL"],
    }
)
X

,Smoker,Shape,Region,Class,Size
0,yes,square,north,third,S
1,no,square,south,first,S
2,no,oval,east,second,L
3,yes,circle,west,third,XL


In [42]:
# Binary encoder
smoker_values = {"no": 0, "yes": 1}
X["Smoker"] = X["Smoker"].map(smoker_values)
X

,Smoker,Shape,Region,Class,Size
0,1,square,north,third,S
1,0,square,south,first,S
2,0,oval,east,second,L
3,1,circle,west,third,XL


In [43]:
# One-hot encoder
# left-to-right column order is alphabetical (circle, oval, square)

ohe = OneHotEncoder(sparse_output=False, drop="first", handle_unknown="ignore") 
ohe_arr = ohe.fit_transform(X[["Shape"]])
ohe_df = pd.DataFrame(ohe_arr, columns=ohe.get_feature_names_out(["Shape"])).astype(int)
X = pd.concat([X, ohe_df], axis=1)
X

,Smoker,Shape,Region,Class,Size,Shape_oval,Shape_square
0,1,square,north,third,S,0,1
1,0,square,south,first,S,0,1
2,0,oval,east,second,L,1,0
3,1,circle,west,third,XL,0,0


In [44]:
# get_dummies
X = pd.get_dummies(X, columns=["Region"], drop_first=True)
X

,Smoker,Shape,Class,Size,Shape_oval,Shape_square,Region_north,Region_south,Region_west
0,1,square,third,S,0,1,True,False,False
1,0,square,first,S,0,1,False,True,False
2,0,oval,second,L,1,0,False,False,False
3,1,circle,third,XL,0,0,False,False,True


In [45]:
# category ordering (within each feature) is defined by you
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[["S", "M", "L", "XL"], ["first", "second", "third"]])
oe_arr = oe.fit_transform(X[["Size", "Class"]])
oe_df = pd.DataFrame(oe_arr, columns=oe.get_feature_names_out(["Size", "Class"])).astype(int)
X = pd.concat([X, oe_df], axis=1)
X

,Smoker,Shape,Class,Size,Shape_oval,Shape_square,Region_north,Region_south,Region_west,Size,Class
0,1,square,third,S,0,1,True,False,False,0,2
1,0,square,first,S,0,1,False,True,False,0,0
2,0,oval,second,L,1,0,False,False,False,2,1
3,1,circle,third,XL,0,0,False,False,True,3,2


### 8. Pipeline

A pipeline is a list of steps, where each step (except the last one) is a **transformer** (must have fit and transform), and the last step can be an **estimator** (e.g. a classifier or regressor that has fit and predict).

#### Pipeline & ColumnTransformer

In [59]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
import numpy as np

In [60]:
np.random.seed(42)

n_samples = 30

data = {
    "age": np.random.randint(20, 65, n_samples),  # 20–64
    "salary": np.random.randint(30000, 90000, n_samples),  # 30k–90k
    "gender": np.random.choice(["M", "F"], n_samples),  
    "city": np.random.choice(["Warsaw", "Krakow", "Gdansk"], n_samples), 
    "target": np.random.choice([0, 1], n_samples, p=[0.4, 0.6])  
}


df = pd.DataFrame(data)
df.head()

,age,salary,gender,city,target
0,58,55658,F,Krakow,1
1,48,48942,M,Krakow,1
2,34,87001,F,Krakow,1
3,62,48431,F,Krakow,0
4,27,32747,F,Gdansk,0


In [65]:
X= df.drop(columns="target")
y = df["target"]

In [66]:
preprocessor = ColumnTransformer([
  ("num", StandardScaler(), ["age", "salary"]),  # numerical features
  ("cat", OneHotEncoder(handle_unknown="ignore"), ["gender", "city"]),   # categorical features
])

pipe = Pipeline([
  ("prep", preprocessor),
  ("clf", LogisticRegression()),  # classifier
])
pipe.fit(X, y)

,steps,"[('prep', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [67]:
# Pipeline z GridSearchCV

from sklearn.model_selection import GridSearchCV

param_grid = {"clf__C": [0.1, 1, 10]}  # parametr modelu

grid = GridSearchCV(pipe, param_grid, cv=2)
grid.fit(X, y)

print("Best parameters:", grid.best_params_)

Best parameters: {'clf__C': 0.1}


#### make_pipeline

In [82]:
from sklearn.pipeline import make_pipeline

train = pd.DataFrame(
    {
        "feat1": [10, 20, np.nan, 2],
        "feat2": [25.0, 20, 5, 3],
        "label": ["A", "A", "B", "B"],
    }
)
test = pd.DataFrame({"feat1": [30.0, 5, 15], "feat2": [12, 10, np.nan]})

In [83]:
imputer = SimpleImputer()
clf = LogisticRegression()

pipe = make_pipeline(imputer, clf)

In [84]:
features = ["feat1", "feat2"]

X, y = train[features], train["label"]
X_new = test[features]

In [85]:
# pipeline applies the imputer to X before fitting the classifier
pipe.fit(X, y)

# pipeline applies the imputer to X_new before making predictions
# note: pipeline uses imputation values learned during the "fit" step
pipe.predict(X_new)

array(['A', 'B', 'A'], dtype=object)

In [86]:
print(pipe.named_steps.simpleimputer.statistics_)
print(pipe.named_steps["logisticregression"].get_params())
print("C =", pipe.named_steps["logisticregression"].C)
print("coef_ =", pipe.named_steps["logisticregression"].coef_)

[10.66666667 13.25      ]
{'C': 1.0, 'class_weight': None, 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': None, 'max_iter': 100, 'multi_class': 'deprecated', 'n_jobs': None, 'penalty': 'l2', 'random_state': None, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
C = 1.0
coef_ = [[-0.16119437 -0.4269462 ]]


### 9. add_missing_indicator

In [73]:
X = pd.DataFrame({"Age": [20, 30, 10, np.nan, 10]})

In [74]:
# impute the mean and add an indicator matrix for missing values
imputer = SimpleImputer(add_indicator=True)
imputer.fit_transform(X)

array([[20. ,  0. ],
       [30. ,  0. ],
       [10. ,  0. ],
       [17.5,  1. ],
       [10. ,  0. ]])

### 11. KNNImputer and IterativeImputer

In [76]:
df = pd.read_csv("http://bit.ly/kaggletrain", nrows=6)
cols = ["SibSp", "Fare", "Age"]
X = df[cols]
X

,SibSp,Fare,Age
0,1,7.2500,22.0
1,1,71.2833,38.0
2,0,7.9250,26.0
3,1,53.1000,35.0
4,0,8.0500,35.0
5,0,8.4583,NaN


In [80]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# IterativeImputer
# Multivariate imputer that estimates each feature from all the others.
# A strategy for imputing missing values by modeling each feature with missing values as
# a function of other features in a round-robin fashion.

imputer = IterativeImputer()
imputer.fit_transform(X)

array([[ 1.        ,  7.25      , 22.        ],
       [ 1.        , 71.2833    , 38.        ],
       [ 0.        ,  7.925     , 26.        ],
       [ 1.        , 53.1       , 35.        ],
       [ 0.        ,  8.05      , 35.        ],
       [ 0.        ,  8.4583    , 28.50639495]])

In [81]:
from sklearn.impute import KNNImputer

# KNNImputer
# Imputation for completing missing values using k-Nearest Neighbors.
# Each sample’s missing values are imputed using the mean value from n_neighbors nearest neighbors 
# found in the training set. Two samples are close if the features that neither is missing are close

imputer = KNNImputer(n_neighbors=2)
imputer.fit_transform(X)

array([[ 1.    ,  7.25  , 22.    ],
       [ 1.    , 71.2833, 38.    ],
       [ 0.    ,  7.925 , 26.    ],
       [ 1.    , 53.1   , 35.    ],
       [ 0.    ,  8.05  , 35.    ],
       [ 0.    ,  8.4583, 30.5   ]])